In [ ]:
import pandas as pd

from utils import create_match_key

In [44]:
arb_candidates = pd.read_json('data/arb_candidates.json')
arb_candidates['platform_ids'].values[0]

{'kalshi': 'KXKHLGAME-26MAR311000SALAVT', 'poly': '307120'}

In [53]:
import requests
import pandas as pd

def hydrate_raw_arbitrage_data(df: pd.DataFrame):
    """
    Fetches raw market data for Poly and Kalshi and returns them as DataFrames.
    No field selection is performed; returns the full API response objects.
    """
    ids_list = df['platform_ids'].tolist()
    poly_ids = [d.get('poly') for d in ids_list if d.get('poly')]
    kalshi_tickers = [d.get('kalshi') for d in ids_list if d.get('kalshi')]

    poly_rows = []
    kalshi_rows = []

    # --- Polymarket (Raw Batch Dump) ---
    if poly_ids:
        try:
            url = "https://gamma-api.polymarket.com/events"
            params = [('id', eid) for eid in poly_ids]
            response = requests.get(url, params=params, timeout=10)
            
            if response.status_code == 200:
                for event in response.json():
                    event_id = str(event.get('id'))
                    for market in event.get('markets', []):
                        # Flattening slightly so we have the parent event ID
                        raw_market = market.copy()
                        raw_market['event_id'] = event_id
                        poly_rows.append(raw_market)
        except Exception as e:
            print(f"Batch Poly error: {e}")

    # --- Kalshi (Raw Individual Dump) ---
    for ticker in kalshi_tickers:
        try:
            url = f"https://api.elections.kalshi.com/trade-api/v2/events/{ticker}?with_nested_markets=true"
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                event_data = response.json().get('event', {})
                parent_ticker = event_data.get('ticker')
                for market in event_data.get('markets', []):
                    raw_market = market.copy()
                    raw_market['event_id'] = parent_ticker
                    kalshi_rows.append(raw_market)
        except Exception as e:
            print(f"Kalshi error for {ticker}: {e}")

    return pd.DataFrame(poly_rows), pd.DataFrame(kalshi_rows)

# Usage:
poly_raw_df, kalshi_raw_df = hydrate_raw_arbitrage_data(arb_candidates)

In [ ]:

poly_raw_df.rename(columns={'id': 'market_id'}, inplace=True)
cols_to_keep = ['event_id', 'market_id', 'question', 'description', 'outcomes', 'outcomePrices', 'bestBid', 'bestAsk']
poly_raw_df = poly_raw_df[cols_to_keep]
poly_raw_df['outcomes']

clean_outcomes = (
    poly_raw_df['outcomes']
    .str.replace(r"[\[\]']", "", regex=True) # Removes [, ], and '
    .str.replace('"', "")                    # Removes "
)


poly_raw_df['descriptive_text'] = (
    poly_raw_df['question'].fillna('') + " " + 
    poly_raw_df['description'].fillna('') + " " + 
    clean_outcomes.fillna('')
).str.strip()



array([['305272', '1746692',
        'Boston Red Sox vs. Cincinnati Reds: O/U 7.5',
        'In the upcoming MLB game between the Boston Red Sox and Cincinnati Reds, scheduled for March 28 at 4:10 PM ET:\n\nThis market will resolve to "Over" if the Boston Red Sox and Cincinnati Reds combine to score 8 or more runs in this game.\n\nIf the combined total is less than 8, this market will resolve to "Under".\n\nIf the game is postponed, this market will remain open until the game has been completed. If the game is canceled entirely, with no make-up game, this market will resolve 50-50.\n\nTo know when a postponed game will be played, please check the home team\'s schedule on MLB.com for the listed team and look for the game described as a makeup game.\n\nThe primary resolution source for this market is the official final statistics of the event as recognized by the governing body or event organizers. However, if the governing body or event organizers have not published final match statisti